# Notebook 12 — Baseline ML Models (Unified Corpus)

**Purpose** TF-IDF + Logistic Regression baselines on the unified EU+US corpus for two tasks: violation-type classification and severity-tier prediction. Includes cross-jurisdiction transfer experiments (train EU → test US and vice versa).

**Inputs**
- `data/unified_cases.csv` (2,754 rows — 337 US, 2,417 EU)

**Outputs**
- Classification reports for both tasks (in-distribution + cross-jurisdiction)

**Key decisions**
- Text feature: `decision_text` (FTC press release for US; GDPRhub facts+holding for EU)
- Violation type target: `coarse_label` (4 classes: Consent / Security / Transparency / Other)
- Severity target: `severity_tier` excluding No Fine (3 classes: Low / Medium / High)
- No Fine excluded from severity: non-monetary enforcement is qualitatively distinct from fined tiers
- Text source differences (press release vs. wiki summary) is a known confounder; transfer results reflect both legal-regime differences and domain shift. If we had real decision texts, this would still be relevant.

In [8]:
#imports
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load unified corpus and sanity-check label coverage.
# coarse_label is missing for ~127 rows (cases whose articles/statutes map to no class);
# severity_tier covers all rows because No Fine is itself a tier.
df = pd.read_csv("/Users/nic/Documents/MM2/data/unified_cases.csv")

print(f"Total rows: {len(df)}")
print(f"US: {(df['jurisdiction'] == 'US').sum()} | EU: {(df['jurisdiction'] != 'US').sum()}")
print(f"\nWith decision_text: {df['decision_text'].notna().sum()}")
print(f"With coarse_label:  {df['coarse_label'].notna().sum()}")
print(f"With severity_tier: {df['severity_tier'].notna().sum()}")

print("\n--- coarse_label distribution ---")
print(df['coarse_label'].value_counts())

print("\n--- severity_tier distribution ---")
print(df['severity_tier'].value_counts())

In [ ]:
# Task 1: Violation Type (coarse_label) — in-distribution 80/20.
# Paper Table VIII comes from this cell's output.
# TF-IDF config (used everywhere in this notebook):
#   max_features=10000  — cap vocabulary at 10k highest-tf-idf terms (not "all features")
#   ngram_range=(1,2)   — unigrams + bigrams (captures phrases like "data breach")
#   sublinear_tf=True   — log-scale term frequency, standard for long legal text
# class_weight='balanced' reweights classes inversely to frequency (Transparency is 3x
# rarer than Consent). stratify keeps class proportions equal across train/test.
task1 = df[df['decision_text'].notna() & df['coarse_label'].notna()].copy()
print(f"Task 1 corpus: {len(task1)} rows")
print(task1['coarse_label'].value_counts())

X = task1['decision_text'].astype(str)
y = task1['coarse_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=777, stratify=y
)

vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tf = vec.fit_transform(X_train)
X_test_tf  = vec.transform(X_test)

clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=777)
clf.fit(X_train_tf, y_train)

print("\n=== Task 1: Violation Type (in-distribution 80/20) ===")
print(classification_report(y_test, clf.predict(X_test_tf)))

In [ ]:
# Task 1 cross-jurisdiction transfer (zero-shot): train on one regime, test on the
# whole other regime. No random split — the partition IS the experiment.
# The vectorizer is refit on each training regime, so the vocabulary is entirely
# EU-derived (or US-derived); unseen test-side terms simply vanish. That is the point:
# this measures whether surface vocabulary transfers. It doesn't (paper Table X).
# Known degenerate result: US-trained model predicts Transparency for 98% of EU cases.
eu = task1[task1['jurisdiction'] != 'US'].copy()
us = task1[task1['jurisdiction'] == 'US'].copy()

print(f"EU: {len(eu)} rows | US: {len(us)} rows")

# Train EU → Test US
vec_eu = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_eu_tf = vec_eu.fit_transform(eu['decision_text'].astype(str))
clf_eu = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf_eu.fit(X_eu_tf, eu['coarse_label'])

X_us_tf = vec_eu.transform(us['decision_text'].astype(str))
print("\n=== Task 1: Train EU → Test US ===")
print(classification_report(us['coarse_label'], clf_eu.predict(X_us_tf)))

# Train US → Test EU
vec_us = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_us_tf2 = vec_us.fit_transform(us['decision_text'].astype(str))
clf_us = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf_us.fit(X_us_tf2, us['coarse_label'])

X_eu_tf2 = vec_us.transform(eu['decision_text'].astype(str))
print("\n=== Task 1: Train US → Test EU ===")
print(classification_report(eu['coarse_label'], clf_us.predict(X_eu_tf2)))

In [ ]:
# Task 2: Severity Tier (Low / Medium / High) — in-distribution 80/20.
# No Fine is excluded: non-monetary enforcement is a different regulatory instrument,
# not a smaller fine (see paper Methodology B).
# NOTE ON SEEDS: this cell (seed 777) prints macro F1 0.52. The paper's Table IX
# (macro 0.55) comes from the text-only condition of the Ruohonen cell below, which
# uses seed 42 on the same task2 corpus. The gap is seed sensitivity, concentrated in
# the 25-support Low tier. Both are honest runs; if you regenerate paper numbers,
# take them from ONE cell consistently.
task2 = df[
    df['decision_text'].notna() &
    df['severity_tier'].notna() &
    (df['severity_tier'] != 'No Fine')
].copy()

print(f"Task 2 corpus: {len(task2)} rows")
print(task2['severity_tier'].value_counts())

X2 = task2['decision_text'].astype(str)
y2 = task2['severity_tier']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=777, stratify=y2
)

vec2 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X2_train_tf = vec2.fit_transform(X2_train)
X2_test_tf  = vec2.transform(X2_test)

clf2 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf2.fit(X2_train_tf, y2_train)

print("\n=== Task 2: Severity Tier (in-distribution 80/20) ===")
print(classification_report(y2_test, clf2.predict(X2_test_tf)))

In [ ]:
# Task 2 cross-jurisdiction transfer (zero-shot), same design as Task 1.
# Structural note: the US test set has NO Low-tier cases (the FTC does not issue
# small fines), so EU→US is effectively a High/Medium problem — that's why its
# macro F1 (0.56) looks deceptively close to in-distribution.
# US→EU collapses: the US-trained model predicts High for all EU cases because
# Low is absent from its training distribution. A finding, not a bug (paper Table X).
eu2 = task2[task2['jurisdiction'] != 'US'].copy()
us2 = task2[task2['jurisdiction'] == 'US'].copy()

print(f"EU: {len(eu2)} rows | US: {len(us2)} rows")

# Train EU → Test US
vec_eu2 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_eu2_tf = vec_eu2.fit_transform(eu2['decision_text'].astype(str))
clf_eu2 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf_eu2.fit(X_eu2_tf, eu2['severity_tier'])

X_us2_tf = vec_eu2.transform(us2['decision_text'].astype(str))
print("\n=== Task 2: Train EU → Test US ===")
print(classification_report(us2['severity_tier'], clf_eu2.predict(X_us2_tf)))

# Train US → Test EU
vec_us2 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_us2_tf2 = vec_us2.fit_transform(us2['decision_text'].astype(str))
clf_us2 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf_us2.fit(X_us2_tf2, us2['severity_tier'])

X_eu2_tf2 = vec_us2.transform(eu2['decision_text'].astype(str))
print("\n=== Task 2: Train US → Test EU ===")
print(classification_report(eu2['severity_tier'], clf_us2.predict(X_eu2_tf2)))

In [ ]:
# Ruohonen challenge: does metadata beat text (their finding) on our corpus?
# Three conditions on the SAME split of task2: metadata-only, text-only, combined.
# Paper Table XI comes from this cell.
# Our metadata is deliberately minimal — jurisdiction (one-hot, 33 countries + US)
# and decision year (numeric). Ruohonen & Hjerppe used 49 variables including cited
# articles and sector; adding GDPR article dummies is the planned enhancement that
# would make the comparison exact.
# Combined = scipy.sparse hstack of the TF-IDF matrix and the metadata matrix —
# one wide feature matrix, same LR on top.
# reset_index first so .loc positions align between the split indices and the frame.
task2_r = task2.reset_index(drop=True)
task2_r['year'] = pd.to_datetime(task2_r['decision_date'], errors='coerce').dt.year.fillna(2020).astype(int)

tr_idx, te_idx, y_tr, y_te = train_test_split(
    task2_r.index.tolist(), task2_r['severity_tier'].tolist(),
    test_size=0.2, random_state=42, stratify=task2_r['severity_tier']
)

# Metadata: jurisdiction (one-hot) + year (numeric)
enc = OneHotEncoder(handle_unknown='ignore')
jur_tr = enc.fit_transform(task2_r.loc[tr_idx, ['jurisdiction']])
jur_te = enc.transform(task2_r.loc[te_idx, ['jurisdiction']])

year_tr = csr_matrix(task2_r.loc[tr_idx, ['year']].values)
year_te = csr_matrix(task2_r.loc[te_idx, ['year']].values)

X_meta_tr = hstack([jur_tr, year_tr])
X_meta_te = hstack([jur_te, year_te])

# Text features
vec3 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_text_tr = vec3.fit_transform(task2_r.loc[tr_idx, 'decision_text'].astype(str))
X_text_te = vec3.transform(task2_r.loc[te_idx, 'decision_text'].astype(str))

# Metadata-only — note Low recall 0.48 vs text's 0.12: jurisdiction partially
# predicts small fines (Low concentrates in specific EU DPAs). See Discussion.
clf_m = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=777)
clf_m.fit(X_meta_tr, y_tr)
print("=== Ruohonen: Metadata-only (jurisdiction + year) ===")
print(classification_report(y_te, clf_m.predict(X_meta_te)))

# Text-only
clf_t = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=777)
clf_t.fit(X_text_tr, y_tr)
print("=== Ruohonen: Text-only ===")
print(classification_report(y_te, clf_t.predict(X_text_te)))

# Combined
clf_c = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=777)
clf_c.fit(hstack([X_text_tr, X_meta_tr]), y_tr)
print("=== Ruohonen: Text + Metadata ===")
print(classification_report(y_te, clf_c.predict(hstack([X_text_te, X_meta_te]))))

## Results Summary

### Task 1: Violation Type (coarse_label)
| Setting | Macro F1 |
|---|---|
| In-distribution (80/20) | 0.75 |
| Train EU → Test US | 0.23 |
| Train US → Test EU | 0.05 |

TF-IDF achieves strong in-distribution performance. Cross-jurisdiction transfer collapses, confirming that surface vocabulary does not transfer across legal regimes. Motivates Legal-BERT as a semantic alternative.

### Task 2: Severity Tier (Low / Medium / High)
| Setting | Macro F1 |
|---|---|
| In-distribution (80/20) | 0.55 |
| Train EU → Test US | 0.56 |
| Train US → Test EU | 0.13 |

### Ruohonen Challenge
| Model | Macro F1 |
|---|---|
| Metadata-only (jurisdiction + year) | 0.46 |
| Text-only (TF-IDF) | 0.55 |
| Text + Metadata | 0.56 |

Text outperforms metadata for severity prediction, partially challenging Ruohonen & Hjerppe (2020). Caveat: their metadata was richer (sector, controller type, articles). Low tier is the exception — jurisdiction predicts Low better than text, consistent with Low being an EU-dominant phenomenon. Combined model adds marginal improvement, suggesting metadata adds little beyond what text already encodes.